# Data Quality Investigation - Interactive Debugging

This notebook allows you to interactively explore the data (like RStudio!).

**Usage:**
1. Run `uv run jupyter notebook` in terminal
2. Open this file in the browser
3. Run cells with Shift+Enter
4. View variables in real-time, modify code, re-run instantly

In [ ]:
# Setup
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd()))

from src.data import load_dataframe
from src.utils.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, CONFIG

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
%matplotlib inline

## 1. Load Data (like loading RData)

In [ ]:
# Load all data
universe = load_dataframe("gcc_banks_universe", RAW_DATA_DIR)
market_data = load_dataframe("prices_daily", RAW_DATA_DIR)
fundamentals = load_dataframe("fundamentals_quarterly", RAW_DATA_DIR)

print(f"Universe: {universe.shape}")
print(f"Market data: {market_data.shape}")
print(f"Fundamentals: {fundamentals.shape}")

In [ ]:
# View universe (like View() in R)
universe.head(20)

## 2. Extract ENBD Data

In [ ]:
# Extract ENBD market data
enbd_market = market_data['ENBD.DU'].copy()
enbd_market.head(10)

In [ ]:
# Missing data summary
print("Missing data for ENBD.DU:\n")
for col in enbd_market.columns:
    missing = enbd_market[col].isna().sum()
    pct = (missing / len(enbd_market)) * 100
    print(f"{col:40s}: {missing:5d} / {len(enbd_market)} ({pct:5.1f}%)")

In [ ]:
# Visualize missing data pattern
fig, axes = plt.subplots(len(enbd_market.columns), 1, figsize=(15, 8))

for idx, col in enumerate(enbd_market.columns):
    ax = axes[idx] if len(enbd_market.columns) > 1 else axes
    
    # Create binary indicator: 1 = data present, 0 = missing
    present = (~enbd_market[col].isna()).astype(int)
    
    ax.fill_between(present.index, 0, present, alpha=0.7)
    ax.set_ylabel(col)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.3)

plt.xlabel('Date')
plt.tight_layout()
plt.show()

## 3. Compare with Quality Thresholds

In [ ]:
# Check quality thresholds from config
print("Quality Thresholds:")
print(f"  Min trading days: {CONFIG.MIN_TRADING_DAYS}")
print(f"  Max missing %: 20%")

# Calculate actual values for ENBD
valid_days = enbd_market['TR.TotalReturn'].notna().sum()
missing_pct = (enbd_market['TR.TotalReturn'].isna().sum() / len(enbd_market)) * 100

print(f"\nENBD.DU Actual:")
print(f"  Valid trading days: {valid_days}")
print(f"  Missing %: {missing_pct:.2f}%")

# Pass/Fail
pass_days = valid_days >= CONFIG.MIN_TRADING_DAYS
pass_missing = missing_pct <= 20

print(f"\nQuality Check:")
print(f"  {'✓' if pass_days else '✗'} Sufficient trading days ({valid_days} >= {CONFIG.MIN_TRADING_DAYS})")
print(f"  {'✓' if pass_missing else '✗'} Acceptable missing rate ({missing_pct:.1f}% <= 20%)")
print(f"\n{'PASSED' if pass_days and pass_missing else 'FAILED'}")

## 4. Explore Fundamentals

In [ ]:
# Extract ENBD fundamentals
enbd_fund = fundamentals['ENBD.DU'].copy()

print(f"Total quarterly records: {len(enbd_fund)}")
print(f"Date range: {enbd_fund.index.min()} to {enbd_fund.index.max()}")
print(f"\nMissing data:")

for col in enbd_fund.columns:
    missing = enbd_fund[col].isna().sum()
    pct = (missing / len(enbd_fund)) * 100
    print(f"  {col:40s}: {missing:3d} / {len(enbd_fund)} ({pct:5.1f}% missing)")

In [ ]:
# View actual fundamental values (non-missing)
enbd_fund.dropna(how='all')

## 5. Compare with Other Banks

Let's see how ENBD compares to other banks

In [ ]:
# Calculate missing % for all banks
banks_missing = []

for bank in universe['bank_ric']:
    try:
        bank_data = market_data[bank]
        ret_col = 'TR.TotalReturn'
        
        if ret_col in bank_data.columns:
            missing_pct = (bank_data[ret_col].isna().sum() / len(bank_data)) * 100
            valid_days = bank_data[ret_col].notna().sum()
            
            banks_missing.append({
                'bank_ric': bank,
                'missing_pct': missing_pct,
                'valid_days': valid_days,
                'pass': (missing_pct <= 20) and (valid_days >= CONFIG.MIN_TRADING_DAYS)
            })
    except KeyError:
        continue

banks_df = pd.DataFrame(banks_missing).sort_values('missing_pct')
banks_df

In [ ]:
# Visualize missing % for all banks
fig, ax = plt.subplots(figsize=(12, 8))

colors = ['green' if p else 'red' for p in banks_df['pass']]
bars = ax.barh(banks_df['bank_ric'], banks_df['missing_pct'], color=colors, alpha=0.6)

# Add threshold line
ax.axvline(x=20, color='red', linestyle='--', linewidth=2, label='20% threshold')

# Highlight ENBD
enbd_idx = banks_df[banks_df['bank_ric'] == 'ENBD.DU'].index[0]
bars[enbd_idx].set_edgecolor('blue')
bars[enbd_idx].set_linewidth(3)

ax.set_xlabel('Missing Data (%)')
ax.set_ylabel('Bank RIC')
ax.set_title('Missing Data % by Bank (ENBD.DU highlighted in blue)')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\nENBD ranking: {list(banks_df['bank_ric']).index('ENBD.DU') + 1} out of {len(banks_df)} banks")

## 6. Interactive Exploration

Add your own cells below to explore further!

In [ ]:
# Your code here - try different things!
# Tip: All variables (universe, market_data, enbd_market, etc.) are available
